# DS603 — Data Structures and Algorithms in Python
## Laboratory 10: Weighted Graph Networks, Greedy Optimization & Disjoint Sets

**Topics:** Topological sorting, directed-cycle detection, Dijkstra's shortest path, Disjoint Set Union (DSU), Prim's and Kruskal's Minimum Spanning Trees, Bellman–Ford forex arbitrage.

### Objective
This notebook implements all five practical problems from the supplied Lab 10 handout. Each problem follows the required three-part structure:
1. Mathematical specification and complexity analysis.
2. Type-annotated Python implementation using standard-library modules only.
3. Verification with sample cases and edge cases.

**Implementation note:** The notebook uses only Python's standard library (`heapq`, `math`, `collections`, and `typing`); no external graph packages are used. Run all cells from top to bottom before submission.

In [1]:
from __future__ import annotations

import heapq
import math
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Set

print("Standard-library imports completed successfully.")

Standard-library imports completed successfully.


---
# Problem 01 — Cloud Build Pipeline & Circular Dependency Audit

## Mathematical specification
A dependency `u → v` means service `u` must finish before service `v` begins. Let `indeg[v]` be the number of incoming edges. Kahn's algorithm repeatedly chooses a zero-indegree vertex; a min-heap ensures the smallest available service ID is chosen first, giving the lexicographically smallest valid order. If fewer than `N` vertices are processed, a directed cycle exists.

For cycle extraction, DFS uses three states: **WHITE** (unvisited), **GRAY** (on the active recursion stack), and **BLACK** (fully explored). An edge to a GRAY vertex identifies a back edge and therefore a cycle.

## Complexity
- Build-order calculation: `O((V + E) log V)` time due to heap operations; `O(V + E)` space.
- Cycle detection / extraction: `O(V + E)` time and space.
- The implementation assumes service IDs are `0 ... N-1` and disallows invalid endpoints.

In [2]:
class BuildPipeline:
    """Dependency planner with lexicographic topological order and cycle extraction."""

    def __init__(self, num_services: int) -> None:
        if num_services < 1:
            raise ValueError("num_services must be at least 1")
        self.n = num_services
        self.adj: List[List[int]] = [[] for _ in range(num_services)]

    def add_dependency(self, u: int, v: int) -> None:
        if not (0 <= u < self.n and 0 <= v < self.n):
            raise IndexError("service ID out of range")
        if u == v:
            raise ValueError("self-dependencies are not allowed")
        self.adj[u].append(v)

    def compute_build_order(self) -> List[int]:
        indegree = [0] * self.n
        for u in range(self.n):
            for v in self.adj[u]:
                indegree[v] += 1

        heap = [v for v in range(self.n) if indegree[v] == 0]
        heapq.heapify(heap)
        order: List[int] = []

        while heap:
            u = heapq.heappop(heap)
            order.append(u)
            for v in self.adj[u]:
                indegree[v] -= 1
                if indegree[v] == 0:
                    heapq.heappush(heap, v)

        if len(order) != self.n:
            raise ValueError("Pipeline deadlock: directed cycle detected")
        return order

    def isolate_cycle(self) -> List[int]:
        WHITE, GRAY, BLACK = 0, 1, 2
        color = [WHITE] * self.n
        parent = [-1] * self.n

        def dfs(u: int) -> List[int]:
            color[u] = GRAY
            for v in self.adj[u]:
                if color[v] == WHITE:
                    parent[v] = u
                    found = dfs(v)
                    if found:
                        return found
                elif color[v] == GRAY:
                    # Reconstruct the cycle from the active ancestor v to u, then close it.
                    chain = [u]
                    while chain[-1] != v:
                        chain.append(parent[chain[-1]])
                    chain.reverse()
                    chain.append(v)
                    return chain
            color[u] = BLACK
            return []

        for vertex in range(self.n):
            if color[vertex] == WHITE:
                found = dfs(vertex)
                if found:
                    return found
        return []

    def has_cycle(self) -> bool:
        return bool(self.isolate_cycle())


In [3]:
# Sample from the handout: valid build pipeline
pipeline = BuildPipeline(6)
for u, v in [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]:
    pipeline.add_dependency(u, v)

assert pipeline.compute_build_order() == [4, 5, 0, 2, 3, 1]
assert pipeline.has_cycle() is False
assert pipeline.isolate_cycle() == []
print("Valid pipeline order:", pipeline.compute_build_order())
print("Has cycle:", pipeline.has_cycle())
print("Cycle path:", pipeline.isolate_cycle())

# Cycle case from the handout
cyclic = BuildPipeline(4)
for edge in [(0, 1), (1, 2), (2, 3), (3, 1)]:
    cyclic.add_dependency(*edge)
assert cyclic.has_cycle()
cycle_path = cyclic.isolate_cycle()
assert cycle_path[0] == cycle_path[-1]
print("Cycle detected:", cyclic.has_cycle())
print("Cycle path:", cycle_path)

# Edge case: isolated single service
single = BuildPipeline(1)
assert single.compute_build_order() == [0]
assert not single.has_cycle()
print("Single-service edge case passed.")

Valid pipeline order: [4, 5, 0, 2, 3, 1]
Has cycle: False
Cycle path: []
Cycle detected: True
Cycle path: [1, 2, 3, 1]
Single-service edge case passed.


---
# Problem 02 — Emergency Dispatch & Turn-by-Turn Route Navigation

## Mathematical specification
The road network is a weighted graph whose edge weights are non-negative travel times. Dijkstra's algorithm maintains a tentative distance `dist[v]` and predecessor `parent[v]`. An edge `(u, v, w)` is relaxed when `dist[u] + w < dist[v]`. The min-heap chooses the smallest tentative distance; stale heap entries are ignored.

`find_fastest_route` returns `(infinity, [])` if the destination is unreachable. `service_radius` returns all vertices whose shortest travel time is at most the specified threshold, sorted by vertex ID.

## Complexity
With a binary heap and adjacency lists: `O((V + E) log V)` time and `O(V + E)` space. Dijkstra is valid only when all road weights are non-negative.

In [4]:
class EmergencyRouter:
    """Find fastest routes on a non-negative weighted road network."""

    def __init__(self, num_intersections: int) -> None:
        if num_intersections < 1:
            raise ValueError("num_intersections must be at least 1")
        self.n = num_intersections
        self.adj: List[List[Tuple[int, float]]] = [
            [] for _ in range(num_intersections)
        ]

    def add_road(
        self, u: int, v: int, travel_time: float, two_way: bool = True
    ) -> None:
        if not (0 <= u < self.n and 0 <= v < self.n):
            raise IndexError("intersection ID out of range")
        if travel_time < 0:
            raise ValueError("Dijkstra requires non-negative travel times")
        self.adj[u].append((v, float(travel_time)))
        if two_way:
            self.adj[v].append((u, float(travel_time)))

    def _dijkstra(self, start: int) -> Tuple[List[float], List[int]]:
        if not 0 <= start < self.n:
            raise IndexError("start intersection out of range")
        dist = [math.inf] * self.n
        parent = [-1] * self.n
        dist[start] = 0.0
        heap = [(0.0, start)]

        while heap:
            current_dist, u = heapq.heappop(heap)
            if current_dist != dist[u]:  # stale heap entry
                continue
            for v, weight in self.adj[u]:
                candidate = current_dist + weight
                if candidate < dist[v]:
                    dist[v] = candidate
                    parent[v] = u
                    heapq.heappush(heap, (candidate, v))
        return dist, parent

    def find_fastest_route(self, start: int, target: int) -> Tuple[float, List[int]]:
        if not 0 <= target < self.n:
            raise IndexError("target intersection out of range")
        dist, parent = self._dijkstra(start)
        if math.isinf(dist[target]):
            return math.inf, []
        path: List[int] = []
        current = target
        while current != -1:
            path.append(current)
            if current == start:
                break
            current = parent[current]
        if not path or path[-1] != start:
            return math.inf, []
        return dist[target], path[::-1]

    def service_radius(self, start: int, max_time: float) -> List[int]:
        if max_time < 0:
            return []
        dist, _ = self._dijkstra(start)
        return sorted(v for v, d in enumerate(dist) if d <= max_time)


In [5]:
router = EmergencyRouter(5)
for u, v, w in [
    (0, 1, 4.0), (0, 2, 2.0), (1, 2, 1.0),
    (1, 3, 5.0), (2, 3, 8.0), (3, 4, 3.0)
]:
    router.add_road(u, v, w)

total_time, route = router.find_fastest_route(0, 4)
assert route == [0, 2, 1, 3, 4]
assert math.isclose(total_time, 11.0)
assert router.service_radius(0, 5.0) == [0, 1, 2]
print("Fastest Route:", route)
print(f"Total Transit Delay: {total_time:.2f} minutes")
print("Reachable within 5.00 mins:", router.service_radius(0, 5.0))

# Edge cases: start equals target; disconnected destination; zero-weight road
assert router.find_fastest_route(2, 2) == (0.0, [2])
isolated = EmergencyRouter(2)
assert isolated.find_fastest_route(0, 1) == (math.inf, [])
zero = EmergencyRouter(2)
zero.add_road(0, 1, 0.0)
assert zero.find_fastest_route(0, 1) == (0.0, [0, 1])
print("Route edge cases passed.")

Fastest Route: [0, 2, 1, 3, 4]
Total Transit Delay: 11.00 minutes
Reachable within 5.00 mins: [0, 1, 2]
Route edge cases passed.


---
# Problem 03 — Dynamic Mesh Connectivity & Redundant Edge Detection (DSU)

## Mathematical specification
A Disjoint Set Union (DSU) maintains connected components. `find(x)` returns the representative (root) of `x`'s component; `union(x, y)` merges two components only when their roots differ. **Path compression** flattens paths during `find`, and **union by rank** attaches the shallower tree below the deeper one. Component sizes and the number of components are maintained.

An edge is redundant if its endpoints already have the same representative before the edge is added.

## Complexity
With path compression and union by rank, a sequence of operations has amortized `O(α(N))` time per operation, where `α` is the inverse Ackermann function (effectively constant for practical input sizes). Space is `O(N)`.

In [6]:
class DisjointSetForest:
    """Union-Find with path compression, union by rank, and component sizes."""

    def __init__(self, n: int) -> None:
        if n < 1:
            raise ValueError("n must be at least 1")
        self.parent = list(range(n))
        self.rank = [0] * n
        self.size = [1] * n
        self.components = n

    def _check(self, x: int) -> None:
        if not 0 <= x < len(self.parent):
            raise IndexError("node ID out of range")

    def find(self, u: int) -> int:
        self._check(u)
        if self.parent[u] != u:
            self.parent[u] = self.find(self.parent[u])
        return self.parent[u]

    def union(self, u: int, v: int) -> bool:
        root_u, root_v = self.find(u), self.find(v)
        if root_u == root_v:
            return False
        if self.rank[root_u] < self.rank[root_v]:
            root_u, root_v = root_v, root_u
        self.parent[root_v] = root_u
        self.size[root_u] += self.size[root_v]
        if self.rank[root_u] == self.rank[root_v]:
            self.rank[root_u] += 1
        self.components -= 1
        return True

    def is_connected(self, u: int, v: int) -> bool:
        return self.find(u) == self.find(v)

    def get_component_size(self, u: int) -> int:
        return self.size[self.find(u)]

    def total_components(self) -> int:
        return self.components


def find_redundant_connection(
    n: int, edges: List[Tuple[int, int]]
) -> Optional[Tuple[int, int]]:
    dsu = DisjointSetForest(n)
    for u, v in edges:
        if not dsu.union(u, v):
            return (u, v)
    return None


In [7]:
dsu = DisjointSetForest(5)
assert dsu.union(0, 1)
assert dsu.union(1, 2)
assert dsu.is_connected(0, 2)
assert dsu.total_components() == 3
assert dsu.get_component_size(1) == 3
assert not dsu.union(0, 2)
assert find_redundant_connection(3, [(0, 1), (1, 2), (0, 2)]) == (0, 2)
print("Connected 0 and 2:", dsu.is_connected(0, 2))
print("Component size containing 1:", dsu.get_component_size(1))
print("Total components:", dsu.total_components())
print("First redundant connection:", find_redundant_connection(3, [(0, 1), (1, 2), (0, 2)]))

# Edge cases: one node; no links; separate components
one = DisjointSetForest(1)
assert one.total_components() == 1 and one.get_component_size(0) == 1
assert find_redundant_connection(3, [(0, 1), (1, 2)]) is None
assert not DisjointSetForest(4).is_connected(0, 3)
print("DSU edge cases passed.")

Connected 0 and 2: True
Component size containing 1: 3
Total components: 3
First redundant connection: (0, 2)
DSU edge cases passed.


---
# Problem 04 — Inter-Datacenter Optical Grid (Dual MST Engine)

## Mathematical specification
A Minimum Spanning Tree (MST) connects every vertex in a connected, undirected, weighted graph using exactly `V - 1` edges with minimum possible total weight.

- **Prim's algorithm** grows one tree. At each step, it selects the cheapest edge crossing from the current tree to a vertex outside it. This follows the cut property.
- **Kruskal's algorithm** sorts all edges by weight and accepts an edge only if its endpoints belong to different DSU components, preventing cycles.

Both algorithms must raise `DisconnectedNetworkException` when not all datacenters can be connected.

## Complexity
- Prim with adjacency lists and a binary heap: `O(E log V)` time, `O(V + E)` space.
- Kruskal: `O(E log E)` time for sorting plus near-linear DSU work; `O(V + E)` space.
- On a connected graph, both algorithms should return the same minimum total cost, though tied weights can produce different valid edge sets.

In [8]:
class DisconnectedNetworkException(Exception):
    """Raised when a spanning tree cannot cover every vertex."""


class FiberGridOptimizer:
    """Minimum-cost cable network using Prim's and Kruskal's algorithms."""

    def __init__(self, num_datacenters: int) -> None:
        if num_datacenters < 1:
            raise ValueError("num_datacenters must be at least 1")
        self.n = num_datacenters
        self.adj: List[List[Tuple[int, float]]] = [
            [] for _ in range(num_datacenters)
        ]
        self.edges: List[Tuple[float, int, int]] = []

    def add_cable_span(self, u: int, v: int, cost: float) -> None:
        if not (0 <= u < self.n and 0 <= v < self.n):
            raise IndexError("datacenter ID out of range")
        if u == v:
            raise ValueError("self-loops are not cable spans")
        if cost <= 0:
            raise ValueError("cable cost must be positive")
        cost = float(cost)
        self.adj[u].append((v, cost))
        self.adj[v].append((u, cost))
        self.edges.append((cost, u, v))

    def prim_mst(self, start: int = 0) -> Tuple[float, List[Tuple[int, int, float]]]:
        if not 0 <= start < self.n:
            raise IndexError("start datacenter out of range")
        visited: Set[int] = {start}
        heap: List[Tuple[float, int, int]] = [
            (cost, start, v) for v, cost in self.adj[start]
        ]
        heapq.heapify(heap)
        selected: List[Tuple[int, int, float]] = []
        total = 0.0

        while heap and len(visited) < self.n:
            cost, u, v = heapq.heappop(heap)
            if v in visited:
                continue
            visited.add(v)
            selected.append((u, v, cost))
            total += cost
            for nxt, nxt_cost in self.adj[v]:
                if nxt not in visited:
                    heapq.heappush(heap, (nxt_cost, v, nxt))

        if len(visited) != self.n:
            raise DisconnectedNetworkException("Network is disconnected")
        return total, selected

    def kruskal_mst(self) -> Tuple[float, List[Tuple[int, int, float]]]:
        dsu = DisjointSetForest(self.n)
        selected: List[Tuple[int, int, float]] = []
        total = 0.0
        for cost, u, v in sorted(self.edges):
            if dsu.union(u, v):
                selected.append((u, v, cost))
                total += cost
                if len(selected) == self.n - 1:
                    break
        if len(selected) != self.n - 1:
            raise DisconnectedNetworkException("Network is disconnected")
        return total, selected


In [9]:
grid = FiberGridOptimizer(5)
for u, v, cost in [
    (0, 1, 1.0), (0, 2, 3.0), (1, 2, 2.0), (1, 3, 4.0),
    (2, 3, 5.0), (2, 4, 6.0), (3, 4, 7.0)
]:
    grid.add_cable_span(u, v, cost)

prim_cost, prim_edges = grid.prim_mst()
kruskal_cost, kruskal_edges = grid.kruskal_mst()
assert len(prim_edges) == 4 and len(kruskal_edges) == 4
assert math.isclose(prim_cost, 13.0) and math.isclose(kruskal_cost, 13.0)
assert math.isclose(prim_cost, kruskal_cost)
print(f"[Prim's MST] Total Cabling Cost: {prim_cost:.2f} M$ | Edges Selected: {len(prim_edges)}")
print("Edges:", prim_edges)
print(f"[Kruskal's MST] Total Cabling Cost: {kruskal_cost:.2f} M$ | Edges Selected: {len(kruskal_edges)}")
print("Edges:", kruskal_edges)
print("Verification: Both MST engines match exactly!")

# Edge cases: disconnected graph should raise; dense graph still yields V-1 edges
disconnected = FiberGridOptimizer(3)
disconnected.add_cable_span(0, 1, 1.0)
for method in (disconnected.prim_mst, disconnected.kruskal_mst):
    try:
        method()
        raise AssertionError("Expected DisconnectedNetworkException")
    except DisconnectedNetworkException:
        pass

dense = FiberGridOptimizer(4)
for u in range(4):
    for v in range(u + 1, 4):
        dense.add_cable_span(u, v, float(u + v + 1))
assert len(dense.prim_mst()[1]) == 3
assert len(dense.kruskal_mst()[1]) == 3
print("Disconnected and dense-graph edge cases passed.")

[Prim's MST] Total Cabling Cost: 13.00 M$ | Edges Selected: 4
Edges: [(0, 1, 1.0), (1, 2, 2.0), (1, 3, 4.0), (2, 4, 6.0)]
[Kruskal's MST] Total Cabling Cost: 13.00 M$ | Edges Selected: 4
Edges: [(0, 1, 1.0), (1, 2, 2.0), (1, 3, 4.0), (2, 4, 6.0)]
Verification: Both MST engines match exactly!
Disconnected and dense-graph edge cases passed.


---
# Problem 05 — Financial Forex Arbitrage Engine (Bellman–Ford Capstone)

## Mathematical specification
For an exchange rate `r` from currency `A` to currency `B`, assign the edge weight `w = -ln(r)`. A cycle is profitable exactly when the product of its exchange rates is greater than 1, which is equivalent to the sum of its transformed weights being negative.

Bellman–Ford starts with distance `0` for every currency (equivalent to a super-source), relaxes all edges up to `V - 1` times, and checks for further relaxation. If an edge can still be relaxed, a negative cycle exists. Following predecessor links `V` times first guarantees entering the cycle; then the cycle can be extracted. `exp(-sum(weights))` converts the cycle's transformed weight back into its exchange-rate multiplier.

## Complexity
Bellman–Ford uses `O(VE)` time and `O(V + E)` space. Currency names are mapped to integer indices internally. All rates must be strictly positive.

In [10]:
class ForexArbitrageEngine:
    """Detect and extract profitable currency conversion cycles."""

    def __init__(self, currencies: List[str]) -> None:
        if len(currencies) < 2 or len(set(currencies)) != len(currencies):
            raise ValueError("Provide at least two unique currencies")
        self.currencies = list(currencies)
        self.index = {currency: i for i, currency in enumerate(currencies)}
        self.edges: List[Tuple[int, int, float]] = []

    def set_exchange_rate(self, from_curr: str, to_curr: str, rate: float) -> None:
        if from_curr not in self.index or to_curr not in self.index:
            raise ValueError("Unknown currency")
        if rate <= 0:
            raise ValueError("Exchange rate must be strictly positive")
        self.edges.append((self.index[from_curr], self.index[to_curr], -math.log(rate)))

    def detect_arbitrage(self) -> Tuple[bool, float, List[str]]:
        n = len(self.currencies)
        dist = [0.0] * n  # super-source: every currency is initially reachable
        parent = [-1] * n
        updated_vertex = -1

        for _ in range(n):
            updated_vertex = -1
            for u, v, weight in self.edges:
                if dist[u] + weight < dist[v] - 1e-15:
                    dist[v] = dist[u] + weight
                    parent[v] = u
                    updated_vertex = v
            if updated_vertex == -1:
                return False, 1.0, []
            # On an early pass without updates, no negative cycle can exist.

        # Move n predecessor steps to guarantee we are inside a cycle.
        inside = updated_vertex
        for _ in range(n):
            inside = parent[inside]
            if inside == -1:
                return False, 1.0, []

        reverse_cycle = [inside]
        current = parent[inside]
        while current != inside and current != -1 and len(reverse_cycle) <= n + 1:
            reverse_cycle.append(current)
            current = parent[current]
        if current != inside:
            return False, 1.0, []
        reverse_cycle.append(inside)
        cycle_indices = list(reversed(reverse_cycle))
        cycle_names = [self.currencies[i] for i in cycle_indices]

        # Compute the exact multiplier using the directed edges in cycle order.
        multiplier = 1.0
        for a, b in zip(cycle_indices, cycle_indices[1:]):
            matching = [
                weight for u, v, weight in self.edges if u == a and v == b
            ]
            if not matching:
                return False, 1.0, []
            # A predecessor relation corresponds to a relaxed edge; use the most
            # profitable matching quote if duplicate quotes were provided.
            multiplier *= math.exp(-min(matching))

        if multiplier <= 1.0 + 1e-12:
            return False, 1.0, []
        return True, multiplier, cycle_names


In [11]:
forex = ForexArbitrageEngine(["USD", "EUR", "GBP"])
for a, b, rate in [
    ("USD", "EUR", 0.90),
    ("EUR", "USD", 1.10),
    ("EUR", "GBP", 0.85),
    ("GBP", "EUR", 1.17),
    ("GBP", "USD", 1.35),
    ("USD", "GBP", 0.73),
]:
    forex.set_exchange_rate(a, b, rate)

found, multiplier, cycle = forex.detect_arbitrage()
assert found is True
assert multiplier > 1.0
assert cycle[0] == cycle[-1]
print("Arbitrage Opportunity Detected:", found)
print("Profitable Cycle:", cycle)
print(f"Net Multiplier: {multiplier:.5f} (Profit: {(multiplier - 1) * 100:+.2f}%)")

# No-arbitrage test: reciprocal rates produce no profitable cycle
balanced = ForexArbitrageEngine(["A", "B"])
balanced.set_exchange_rate("A", "B", 2.0)
balanced.set_exchange_rate("B", "A", 0.5)
found2, multiplier2, cycle2 = balanced.detect_arbitrage()
assert found2 is False and multiplier2 == 1.0 and cycle2 == []
print("No-arbitrage edge case:", (found2, multiplier2, cycle2))

# Invalid rates must be rejected
try:
    balanced.set_exchange_rate("A", "B", 0.0)
    raise AssertionError("Expected ValueError for zero exchange rate")
except ValueError:
    pass
print("Forex edge cases passed.")

Arbitrage Opportunity Detected: True
Profitable Cycle: ['USD', 'EUR', 'GBP', 'USD']
Net Multiplier: 1.03275 (Profit: +3.28%)
No-arbitrage edge case: (False, 1.0, [])
Forex edge cases passed.


---
# Final Summary and Submission Checklist

## Algorithms implemented
| Problem | Algorithm / Data Structure | Time Complexity |
|---|---|---|
| 1. Build pipeline | Kahn's algorithm + 3-color DFS | `O((V+E) log V)` for order; `O(V+E)` cycle detection |
| 2. Emergency routing | Dijkstra + min-heap | `O((V+E) log V)` |
| 3. Mesh connectivity | DSU with path compression and union by rank | Amortized `O(α(V))` per operation |
| 4. Optical grid | Prim + Kruskal + DSU | Prim `O(E log V)`; Kruskal `O(E log E)` |
| 5. Forex arbitrage | Bellman–Ford + negative-cycle extraction | `O(VE)` |
